# Tuesday MVP — Two-Agent Research Pipeline (Supervisor Pattern)

Reads a research question, uses **Agent 1** to pull grounded findings from the materials-science corpus,
hands those findings to **Agent 2** to propose 3 experiment hypotheses. Two agents, one deterministic
router between them — that router is the "supervisor," not a third LLM.

The heavy code lives in `research_pipeline.py` (imported below). This notebook is the walkthrough:
what each piece is, why it's shaped this way, and the pipeline running end-to-end on one real question.

## 0. What's already built — imported, not rebuilt

Three weeks of knowledge-layer work, reused as-is:

- **`materials_rag.py`** — PDFs → chunks → OpenAI embeddings → **persisted** Chroma (`./chroma_db`,
  collection `materials_papers`). Chunks carry `paper` / `section_type` / `year` / `authors` /
  `method_type` metadata (see `PAPER_META` + `tag_metadata()` — rebuild with
  `uv run python materials_rag.py --reindex` if you ever change that catalog or the chunking).
- **`search_papers`** — the `@tool`-wrapped retriever. Agent 1 calls this; it never touches Chroma,
  embeddings, or chunking directly.

This notebook only adds the two-agent pipeline on top.

In [1]:
import research_pipeline as rp
from materials_rag import vectorstore

print(f"Persisted chunks in ./chroma_db: {vectorstore._collection.count()}")
sample = vectorstore.get(limit=1, include=["metadatas"])["metadatas"][0]
print("Sample chunk metadata:", sample)

Loading existing index from ./chroma_db


Persisted chunks in ./chroma_db: 1070
Sample chunk metadata: {'keywords': '', 'page_label': '1', 'paper': 'chemcrow', 'title': '', 'page': 0, 'subject': '', 'moddate': '2023-10-03T01:55:50+00:00', 'method_type': 'LLM agent', 'producer': 'pdfTeX-1.40.25', 'creationdate': '2023-10-03T01:55:50+00:00', 'year': 2023, 'source': 'papers/chemcrow.pdf', 'start_index': 0, 'venue': 'Nature Machine Intelligence', 'creator': 'LaTeX with hyperref', 'trapped': '/False', 'authors': 'Bran et al.', 'total_pages': 38, 'author': '', 'section_type': 'header', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5'}


## 1. Supervisor, not swarm — why

A **swarm** lets agents hand off to each other freely — good for open-ended, parallel exploration.
This task is a **dependent pipeline**: you can't hypothesize before you know what the literature says.
That's a supervisor shape — one controller, agents run in a fixed order, decisions are made by looking
at shared state, not by an LLM improvising who goes next.

This matches Anthropic's orchestrator-worker pattern and sidesteps what Cognition's *"Don't Build
Multi-Agents"* warns about: peer agents stepping on each other's context with no single source of
truth. It's also designed against two failure modes from Cemri et al., *"Why Do Multi-Agent LLM Systems
Fail?"* (arXiv 2503.13657) — **absent termination criteria** and **incomplete context propagation across
the handoff** — covered in detail in §5 and §9 below.

## 2. State-as-handoff: the shared schema

Agent 2 never reads Agent 1's chat transcript (tool calls, retries, intermediate reasoning). It reads
**typed fields on shared state** — the handoff is data, not messages. `PipelineState` is the whole
contract:

In [2]:
import inspect
print(inspect.getsource(rp.PipelineState))

class PipelineState(TypedDict, total=False):
    question: str
    literature_findings: dict     # LiteratureFindings.model_dump()
    hypotheses: list[dict]        # [Hypothesis.model_dump(), ...]
    status: str                   # how/why the run ended — always set on a terminal node



Each agent's output is a **Pydantic model**, not free text — `create_agent(..., response_format=Model)`
re-prompts the model until its output actually validates against the schema. That's what makes
"exactly 3 hypotheses" a structural guarantee (`min_length=3, max_length=3` below) instead of a prompt
instruction the model can quietly ignore.

In [3]:
print(inspect.getsource(rp.PaperSummary))
print(inspect.getsource(rp.LiteratureFindings))
print(inspect.getsource(rp.Hypothesis))
print(inspect.getsource(rp.SynthesisOutput))

class PaperSummary(BaseModel):
    paper: str = Field(description="Paper stem, e.g. 'mace-mp-0' — read from the retrieved chunk's metadata tag.")
    key_claim: str = Field(description="The paper's main claim relevant to the research question.")
    method: str = Field(description="The method/approach the paper uses.")
    relevant_finding: str = Field(description="The specific result, number, or finding relevant to the question.")
    source_citation: str = Field(description="Paper + page (from the metadata tag) this summary is grounded in.")

class LiteratureFindings(BaseModel):
    summaries: list[PaperSummary] = Field(
        default_factory=list,
        description="One entry per distinct paper with retrieved evidence. Empty if nothing relevant was found — "
                     "do not invent a summary for a paper you didn't actually retrieve.",
    )

class Hypothesis(BaseModel):
    hypothesis: str = Field(description="A concrete, testable experiment hypothesis.")
    variabl

## 3. Agent 1 — Literature agent

A `create_agent` ReAct loop with exactly one tool: `search_papers`. It decides how many times to search
and when it's covered enough ground — that tool-calling judgment is exactly what `create_agent` is for.
What it must NOT decide is its output *shape*: `response_format=LiteratureFindings` pins that down.

In [4]:
print(rp._LITERATURE_SYSTEM_PROMPT)

You are the literature-review agent in a two-agent materials-science research pipeline. Use the search_papers tool to find passages relevant to the research question — call it more than once with different phrasings if the first query doesn't surface enough, but stop once you've covered the relevant papers (don't search indefinitely). Each retrieved passage is tagged with its paper, section, year, authors, and method_type — use those tags, don't guess them.

Produce exactly one structured summary per DISTINCT paper you found solid evidence for. Never summarize a paper you didn't actually retrieve a passage from. If nothing relevant turns up, return an empty summaries list rather than forcing an answer.


## 4. Agent 2 — Synthesis agent

No tools at all — it can only reason over what Agent 1 handed it in state. That's deliberate: giving it
`search_papers` too would blur the pipeline back toward a swarm (either agent could go re-retrieve,
so "who's responsible for grounding" stops being answerable from the code).

In [5]:
print(rp._SYNTHESIS_SYSTEM_PROMPT)

You are the synthesis agent in a two-agent materials-science research pipeline. You do not have tools and cannot search — you only see the literature findings handed to you below. Ground every hypothesis in those findings; do not introduce papers, numbers, or claims that aren't in them.

Propose exactly 3 concrete, testable experiment hypotheses. For each: the hypothesis itself, the variables an experiment would manipulate/measure, the expected outcome, and a confidence score. confidence is YOUR OWN self-reported estimate, not a calibrated probability — treat it as a rough prior.


## 5. The handoff, and where the "supervisor" actually lives

Between the two agents sits one function: a routing decision on shared state, not an LLM call.
This *is* the supervisor — routing logic, not a third agent burning latency/cost on a call whose
answer is just "is this list empty?" This is also the **explicit termination criterion**: if Agent 1
found nothing, the graph ends at `end_no_papers` instead of handing Agent 2 an empty context and
letting it hallucinate hypotheses out of thin air.

In [6]:
print(inspect.getsource(rp.route_after_literature))
print(inspect.getsource(rp._format_findings_for_handoff))

def route_after_literature(state: PipelineState) -> Literal["synthesis", "end_no_papers"]:
    """The supervisor's one routing decision — deterministic, not an LLM's judgment call."""
    if not state["literature_findings"]["summaries"]:
        return "end_no_papers"
    return "synthesis"

def _format_findings_for_handoff(findings: dict) -> str:
    """Render Agent 1's FULL structured output into Agent 2's prompt — every field
    the schema captured, not a shortened re-summary. See module docstring: this is
    the guard against lossy handoff, not decoration."""
    return "\n\n".join(
        f"- [{s['paper']}] {s['key_claim']}\n"
        f"    method: {s['method']}\n"
        f"    finding: {s['relevant_finding']}\n"
        f"    source: {s['source_citation']}"
        for s in findings["summaries"]
    )



`_format_findings_for_handoff` is the other half of "handoff = state, not messages": it renders
**every field** of Agent 1's validated output into Agent 2's prompt — not a shortened re-summary.
That's the direct guard against Cemri et al.'s "incomplete context propagation" failure mode.

## 6. Graph wiring

Four nodes, every path reaches `END`:

In [7]:
graph = rp.build_graph()
print(graph.get_graph().draw_mermaid())
# ^ paste into https://mermaid.live if your notebook viewer doesn't render mermaid inline

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	literature(literature)
	synthesis(synthesis)
	end_no_papers(end_no_papers)
	human_review(human_review)
	__end__([<p>__end__</p>]):::last
	__start__ --> literature;
	literature -.-> end_no_papers;
	literature -.-> synthesis;
	synthesis --> human_review;
	end_no_papers --> __end__;
	human_review --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 7. Checkpointer + thread_id

`build_graph()` compiles with a real checkpointer (`InMemorySaver` — swap for `PostgresSaver` outside
a demo) so state, and later the interrupt in §10, survive across calls on the same `thread_id`.

**Gotcha to respect:** plain `TypedDict` fields persist across `.invoke()` calls on the *same*
`thread_id`. Reuse a `thread_id` for a second, unrelated question and `literature_findings` /
`hypotheses` from the FIRST question can leak into the second run's state — the same dangling-state
trap `agentic_rag.py` already documents. `run_pipeline()` mints a fresh `uuid` thread_id by default;
only pass one explicitly for a genuine follow-up on the same question.

In [8]:
print(inspect.getsource(rp.run_pipeline))

def run_pipeline(question: str, graph=None, thread_id: str | None = None) -> PipelineState:
    """Mints a fresh thread_id per question by default — same gotcha as
    agentic_rag.py: reusing a thread_id across unrelated questions lets stale
    literature_findings/hypotheses from a PREVIOUS question leak into this run's
    state, since plain (non-Annotated) TypedDict fields persist across .invoke()
    calls on the same thread. Only pass thread_id explicitly for a genuine
    follow-up on the same question thread.

    NOTE: if the graph pauses at human_review, the returned dict is state-so-far
    with an extra "__interrupt__" key, not a finished run — this function does
    not resume it (it doesn't expose the thread_id needed to). For an
    interactive run that can actually resume, see __main__ below or app.py's
    Chainlit UI, both of which keep the thread_id around after the pause."""
    graph = graph or build_graph()
    config = {"configurable": {"thread_id": thread_id or s

## 8. Run it end-to-end

One question from your corpus. Streaming `stream_mode="updates"` shows each node firing in order —
watch `literature` do its retrieve/summarize work before `synthesis` ever runs; that ordering is the
whole point of "supervisor," not "swarm." The run **pauses right after `synthesis`**, at
`human_review` — a real `interrupt()`, not a stub — and stays paused until §10 resumes it on this
same `thread_id`.

In [9]:
import uuid

QUESTION = (
    "What formation-energy prediction error do current foundation potentials "
    "(MACE-MP-0, Orb-v3, UMA) report, and where might a new benchmark or "
    "architecture change close the remaining gap?"
)

thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

for update in graph.stream({"question": QUESTION}, config=config, stream_mode="updates"):
    for node_name, node_output in update.items():
        if node_name == "__interrupt__":
            # human_review's interrupt() call — the graph is paused HERE, not finished.
            print("--- interrupt: human_review (paused — resumed in §10) ---\n")
            continue
        print(f"--- node: {node_name} ---")
        if node_name == "literature":
            for s in node_output["literature_findings"]["summaries"]:
                print(f"  [{s['paper']}] {s['key_claim'][:100]}")
        elif node_name == "synthesis":
            print(f"  {len(node_output['hypotheses'])} hypotheses generated")
        else:
            print(f"  {node_output}")
        print()

# Paused at human_review — this is state-SO-FAR, not the finished run.
final_state = graph.get_state(config).values
print("graph.get_state(config).next:", graph.get_state(config).next)

--- node: literature ---
  [mace-mp-0] MACE foundation potentials generalize to out-of-distribution hypothetical materials, but formation-e
  [orb3] Orb-v3 changes the architecture/performance trade-off for foundation potentials, showing that non-eq
  [universalmodelofatoms] UMA improves broad-domain foundation-potential accuracy through very large-scale training and a mixt
  [omat-2024] A new benchmark/data protocol can expose and potentially reduce remaining formation-energy and hull-



--- node: synthesis ---
  3 hypotheses generated

--- interrupt: human_review (paused — resumed in §10) ---

graph.get_state(config).next: ('human_review',)


## 9. Hypotheses proposed by Agent 2 (pending your review)

Not final yet — the graph is paused at `human_review`. This is what a reviewer would see.

In [10]:
print(f"Question: {QUESTION}\n")
print(f"Papers grounding this: {[s['paper'] for s in final_state['literature_findings']['summaries']]}\n")

for i, h in enumerate(final_state["hypotheses"], 1):
    print(f"{i}. {h['hypothesis']}")
    print(f"   variables: {h['variables']}")
    print(f"   expected outcome: {h['expected_outcome']}")
    print(f"   confidence (SELF-REPORTED — model's own prior, not calibrated): {h['confidence']}")
    print()

Question: What formation-energy prediction error do current foundation potentials (MACE-MP-0, Orb-v3, UMA) report, and where might a new benchmark or architecture change close the remaining gap?

Papers grounding this: ['mace-mp-0', 'orb3', 'universalmodelofatoms', 'omat-2024']

1. A validation-set affine calibration of MACE-MPA-0 formation energies will reduce its WBM formation-energy error by correcting the reported systematic underprediction.
   variables: ['Model: MACE-MP-0b3 vs MACE-MPA-0', 'Calibration condition: no correction vs affine correction fitted on a held-out validation subset', 'Measured metrics: WBM formation-energy MAE, RMSE, R², residual mean bias, fitted slope/intercept of predicted vs DFT PBE formation energy', 'Optional downstream metric: Matbench Discovery stability classification/F1 after calibrated single-point energies']
   expected outcome: Compared with the uncorrected MACE-MPA-0 WBM MAE of 28 meV/atom and its reported least-squares trend y=0.9x−0.22, the af

## 10. The human-in-the-loop gate — implemented, not stubbed

`human_review_node` sits between `synthesis` and `END`. It's a real `langgraph.types.interrupt()` call
— the trace in §8 shows it firing and pausing, and `graph.get_state(config).next` above confirms
`('human_review',)` is still pending right now.

In [11]:
print(inspect.getsource(rp.human_review_node))
print(inspect.getsource(rp.apply_review_decision))

def human_review_node(state: PipelineState) -> dict:
    """Pauses the graph for a human to approve, edit, or reject each of Agent 2's
    3 hypotheses. See the module docstring's "Human-in-the-loop gate" section for
    the interrupt/resume contract and the thread_id gotcha.

    The interrupt payload hands the reviewer state["hypotheses"] VERBATIM — same
    "full validated data, not a summary" rule as the literature->synthesis
    handoff. A caller resumes with Command(resume=decision) where decision is
    {"approved_hypotheses": [...], "status": "<free text>"}; approved_hypotheses
    becomes the pipeline's final hypotheses list, and status overwrites
    synthesis_node's "complete" with whatever the reviewer/caller reports (e.g.
    "reviewed", "review timed out — treated as reject").
    """
    decision = interrupt({
        "hypotheses": state["hypotheses"],
        "action": "approve, edit, or reject each hypothesis",
    })
    return {
        "hypotheses": decision.get("ap

Resuming needs `Command(resume=decision)` on the **SAME `thread_id`** used in §8 — a different
`thread_id`, or one whose last step was a half-finished tool call, corrupts checkpointer state (see
`human_review_node`'s docstring above). `decision` is
`{"approved_hypotheses": [...], "status": "<free text>"}`. `apply_review_decision` is the shared
approve/edit/reject text convention this notebook, the CLI (`research_pipeline.py`'s `__main__`), and
`app.py`'s Chainlit UI all use to build `approved_hypotheses` from a plain-text reply:
- `"all"` — keep every hypothesis unchanged
- `"none"` (or empty) — reject everything
- `"1, 3"` — keep only hypotheses #1 and #3, drop #2
- `"2: revised hypothesis text"` — keep #2 but replace its text before approving

In [12]:
from langgraph.types import Command

REVIEW_REPLY = "1, 3"  # demo decision: keep #1 and #3, reject #2 (edit syntax: "2: revised text")

approved = rp.apply_review_decision(final_state["hypotheses"], REVIEW_REPLY)
final_state = graph.invoke(
    Command(resume={"approved_hypotheses": approved, "status": "reviewed in notebook"}),
    config=config,
)

print("status:", final_state["status"])
print("next pending node:", graph.get_state(config).next, "(empty tuple = reached END)\n")
print(f"{len(final_state['hypotheses'])} hypothesis(es) kept after review:\n")
for i, h in enumerate(final_state["hypotheses"], 1):
    print(f"{i}. {h['hypothesis']}")
    print(f"   confidence (self-reported): {h['confidence']}\n")

status: reviewed in notebook
next pending node: () (empty tuple = reached END)

2 hypothesis(es) kept after review:

1. A validation-set affine calibration of MACE-MPA-0 formation energies will reduce its WBM formation-energy error by correcting the reported systematic underprediction.
   confidence (self-reported): 0.78

2. A benchmark protocol that enforces DFT-setting consistency, Materials-Project-style compatibility corrections, and unique-prototype splits will expose larger out-of-distribution formation/hull-energy errors than less stringent evaluations, and models fine-tuned or calibrated to those settings will reduce the gap.
   confidence (self-reported): 0.7



## 11. What this design guards against — and what it doesn't

**Guards against (Cemri et al. 2503.13657):**
- *Absent termination criteria* → `route_after_literature` ends the graph explicitly when Agent 1 finds
  nothing, instead of looping or forcing Agent 2 to invent hypotheses from an empty context.
- *Incomplete context propagation across the handoff* → Agent 2 gets Agent 1's full validated
  `LiteratureFindings`, not a chat transcript or a shortened summary.
- *Ungrounded output reaching the end user unchecked* (partially) → `human_review` (§10) puts every
  hypothesis in front of a person, verbatim, before it's final — they can edit or reject each one.

**Does NOT yet guard against (flagged, not solved):**
- The human review gate is a **manual** check, not an automatic one — nothing stops a reviewer from
  replying `"all"` without reading. There is still no automatic faithfulness/entailment pass, unlike
  `agentic_rag.py`'s `verify_faithfulness` step, for whatever the human approves.
- Agent 1's own tool-calling loop has no hard cap beyond LangGraph's default `recursion_limit` (25) on
  that agent's internal graph — bounded by the system prompt's judgment, not enforced in code.
- No retry/escalation policy if `response_format` validation keeps failing.

**Kept to two agents, on purpose.** A "critic" agent to catch the faithfulness gap above, or a "router"
agent to replace `route_after_literature`, would both be tempting — flagging both as future work rather
than building them today.